# Model 3 — Independent Static GCN for LADI-v2

# LADI-v2 independent Kaggle notebook

This notebook is **fully self-contained**. It does not depend on outputs from notebook `00`, `01`, or any other notebook.

It will:

1. install/check requirements,
2. load the LADI-v2 dataset directly from Hugging Face,
3. prepare train/validation/test dataloaders,
4. train the selected model,
5. tune validation thresholds,
6. evaluate on validation and test,
7. save checkpoint, predictions, metrics, and configuration under `/kaggle/working/`.

**Kaggle setting:** GPU P100, Internet ON.

The default Hugging Face branch loads the compact `v2a` label set with resized images. If you only want a smoke test, set `LIMIT_TRAIN`, `LIMIT_VAL`, and `LIMIT_TEST` below.


In [ ]:
# ============================================================
# 0. Requirements: self-contained installation cell
# ============================================================
# Kaggle usually already has torch/torchvision. We install only lightweight missing packages.
!pip install -q datasets scikit-learn pandas matplotlib tqdm pillow


In [ ]:

# ============================================================
# 1. Imports, configuration, reproducibility
# ============================================================
import os, json, math, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import torchvision.transforms as T
import torchvision.models as models
from datasets import load_dataset
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -------------------- editable config --------------------
DATASET_NAME = "MITLL/LADI-v2-dataset"
OUTPUT_ROOT = Path("/kaggle/working/ladi_independent_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# For a fast smoke test, set e.g. LIMIT_TRAIN=512, LIMIT_VAL=128, LIMIT_TEST=128.
# For the real experiment, keep them as None.
LIMIT_TRAIN = None
LIMIT_VAL = None
LIMIT_TEST = None

IMAGE_SIZE = 320       # P100-safe. If OOM: try 288 or 224.
BATCH_SIZE = 16        # Dynamic GCN notebook may override this to 8 or 12.
NUM_WORKERS = 2
NUM_EPOCHS = 12        # Increase to 20-30 for final runs.
LR = 2e-4
WEIGHT_DECAY = 1e-4
USE_AMP = True
PATIENCE = 4

LABEL_COLS = [
    "bridges_any",
    "buildings_any",
    "buildings_affected_or_greater",
    "buildings_minor_or_greater",
    "debris_any",
    "flooding_any",
    "flooding_structures",
    "roads_any",
    "roads_damage",
    "trees_any",
    "trees_damage",
    "water_any",
]
NUM_LABELS = len(LABEL_COLS)
print("Labels:", NUM_LABELS, LABEL_COLS)

# RUN_NAME is set by each notebook.


# -------------------- model-specific config --------------------
RUN_NAME = "model_03_static_gcn_resnet18_independent"
BACKBONE_NAME = "resnet18"
GRAPH_TYPE = "pmi"      # options: pmi, condprob, cooc, identity
GRAPH_TOP_K = 5
LABEL_EMBED_DIM = 256
GCN_HIDDEN_DIM = 512
DROPOUT = 0.25
CONFIG = {
    "run_name": RUN_NAME,
    "model": "StaticGCNClassifier",
    "backbone": BACKBONE_NAME,
    "graph_type": GRAPH_TYPE,
    "graph_top_k": GRAPH_TOP_K,
    "label_embed_dim": LABEL_EMBED_DIM,
    "gcn_hidden_dim": GCN_HIDDEN_DIM,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "limit_train": LIMIT_TRAIN,
    "limit_val": LIMIT_VAL,
    "limit_test": LIMIT_TEST,
    "label_cols": LABEL_COLS,
}
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# ============================================================
# 2. Dataset loading and dataloaders: included inside this notebook
# ============================================================
def subset_hf_dataset(ds_split, limit):
    if limit is None:
        return ds_split
    limit = min(limit, len(ds_split))
    return ds_split.select(range(limit))

print("Loading LADI-v2 from Hugging Face...")
# Default config loads v2a with resized images according to the HF dataset card.
ds = load_dataset(DATASET_NAME)
print(ds)

train_hf = subset_hf_dataset(ds["train"], LIMIT_TRAIN)
val_hf = subset_hf_dataset(ds["validation"], LIMIT_VAL)
test_hf = subset_hf_dataset(ds["test"], LIMIT_TEST)

print("Split sizes:", len(train_hf), len(val_hf), len(test_hf))

def get_labels_matrix(hf_split):
    rows = []
    for ex in tqdm(hf_split, desc="Collecting labels"):
        rows.append([float(bool(ex[c])) for c in LABEL_COLS])
    return np.asarray(rows, dtype=np.float32)

Y_train_np = get_labels_matrix(train_hf)
Y_val_np = get_labels_matrix(val_hf)
Y_test_np = get_labels_matrix(test_hf)

print("Training label prevalence:")
for name, prev in zip(LABEL_COLS, Y_train_np.mean(axis=0)):
    print(f"  {name:34s}: {prev:.4f}")

# ImageNet transforms; CNN pretrained weights expect this normalization.
train_tfms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02)], p=0.35),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class LADIMultiLabelDataset(Dataset):
    def __init__(self, hf_split, transform, label_cols):
        self.hf_split = hf_split
        self.transform = transform
        self.label_cols = label_cols
    def __len__(self):
        return len(self.hf_split)
    def __getitem__(self, idx):
        ex = self.hf_split[int(idx)]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.open(img)
        img = img.convert("RGB")
        x = self.transform(img)
        y = torch.tensor([float(bool(ex[c])) for c in self.label_cols], dtype=torch.float32)
        return x, y, int(idx)

train_loader = DataLoader(
    LADIMultiLabelDataset(train_hf, train_tfms, LABEL_COLS),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    LADIMultiLabelDataset(val_hf, eval_tfms, LABEL_COLS),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    LADIMultiLabelDataset(test_hf, eval_tfms, LABEL_COLS),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

# Class imbalance weights for BCEWithLogitsLoss.
pos = Y_train_np.sum(axis=0)
neg = len(Y_train_np) - pos
pos_weight_np = neg / np.maximum(pos, 1.0)
pos_weight_np = np.clip(pos_weight_np, 1.0, 20.0).astype(np.float32)
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32, device=device)
print("pos_weight:", dict(zip(LABEL_COLS, pos_weight_np.round(2))))

# ============================================================
# 3. Metrics, threshold tuning, train/evaluate helpers
# ============================================================
def safe_average_precision(y_true, y_score):
    vals = []
    for j in range(y_true.shape[1]):
        if y_true[:, j].sum() == 0:
            continue
        vals.append(average_precision_score(y_true[:, j], y_score[:, j]))
    return float(np.mean(vals)) if vals else 0.0

def multilabel_metrics(y_true, y_prob, thresholds=None):
    if thresholds is None:
        thresholds = np.full(y_true.shape[1], 0.5, dtype=np.float32)
    y_pred = (y_prob >= thresholds[None, :]).astype(np.int32)
    out = {
        "macro_ap": safe_average_precision(y_true, y_prob),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y_true, y_pred, average="micro", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    per_label = []
    for j, name in enumerate(LABEL_COLS):
        ap = average_precision_score(y_true[:, j], y_prob[:, j]) if y_true[:, j].sum() > 0 else 0.0
        per_label.append({
            "label": name,
            "prevalence": float(y_true[:, j].mean()),
            "ap": float(ap),
            "f1": float(f1_score(y_true[:, j], y_pred[:, j], zero_division=0)),
            "threshold": float(thresholds[j]),
        })
    return out, pd.DataFrame(per_label)

def tune_thresholds(y_true, y_prob):
    grid = np.arange(0.05, 0.96, 0.05)
    th = np.zeros(y_true.shape[1], dtype=np.float32)
    for j in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            pred = (y_prob[:, j] >= t).astype(np.int32)
            f1 = f1_score(y_true[:, j], pred, zero_division=0)
            if f1 > best_f1:
                best_t, best_f1 = float(t), float(f1)
        th[j] = best_t
    return th

@torch.no_grad()
def predict_loader(model, loader):
    model.eval()
    probs, targets, indices = [], [], []
    for x, y, idx in tqdm(loader, desc="Predict", leave=False):
        x = x.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP and device.type == "cuda"):
            logits = model(x)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        targets.append(y.numpy())
        indices.append(np.asarray(idx))
    return np.vstack(probs), np.vstack(targets), np.concatenate(indices)

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    n = 0
    for x, y, _ in tqdm(loader, desc="Train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=USE_AMP and device.type == "cuda"):
            logits = model(x)
            loss = criterion(logits, y)
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        bs = x.size(0)
        total_loss += float(loss.detach().cpu()) * bs
        n += bs
    return total_loss / max(n, 1)

def save_predictions_csv(path, probs, y_true, indices, thresholds):
    rows = []
    y_pred = (probs >= thresholds[None, :]).astype(np.int32)
    for i in range(len(probs)):
        row = {"index": int(indices[i])}
        for j, name in enumerate(LABEL_COLS):
            row[f"true_{name}"] = int(y_true[i, j])
            row[f"prob_{name}"] = float(probs[i, j])
            row[f"pred_{name}"] = int(y_pred[i, j])
        rows.append(row)
    pd.DataFrame(rows).to_csv(path, index=False)

def run_training(model, run_dir):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    scaler = GradScaler(enabled=USE_AMP and device.type == "cuda")

    best_ap = -1.0
    best_epoch = -1
    bad_epochs = 0
    history = []

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        scheduler.step()
        val_prob, val_true, val_idx = predict_loader(model, val_loader)
        val_metrics, _ = multilabel_metrics(val_true, val_prob, thresholds=None)
        print({"train_loss": round(loss, 4), **{k: round(v, 4) for k, v in val_metrics.items()}})
        history.append({"epoch": epoch, "train_loss": loss, **val_metrics})

        if val_metrics["macro_ap"] > best_ap:
            best_ap = val_metrics["macro_ap"]
            best_epoch = epoch
            bad_epochs = 0
            torch.save({
                "model_state_dict": model.state_dict(),
                "label_cols": LABEL_COLS,
                "config": CONFIG,
                "best_val_macro_ap": best_ap,
                "best_epoch": best_epoch,
            }, run_dir / "best_model.pt")
            print("Saved new best checkpoint.")
        else:
            bad_epochs += 1
            print(f"No improvement. bad_epochs={bad_epochs}/{PATIENCE}")
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
    checkpoint = torch.load(run_dir / "best_model.pt", map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    # Tune thresholds on validation only.
    val_prob, val_true, val_idx = predict_loader(model, val_loader)
    thresholds = tune_thresholds(val_true, val_prob)
    val_metrics, val_per_label = multilabel_metrics(val_true, val_prob, thresholds)

    test_prob, test_true, test_idx = predict_loader(model, test_loader)
    test_metrics, test_per_label = multilabel_metrics(test_true, test_prob, thresholds)

    save_predictions_csv(run_dir / "val_predictions.csv", val_prob, val_true, val_idx, thresholds)
    save_predictions_csv(run_dir / "test_predictions.csv", test_prob, test_true, test_idx, thresholds)
    val_per_label.to_csv(run_dir / "val_per_label_metrics.csv", index=False)
    test_per_label.to_csv(run_dir / "test_per_label_metrics.csv", index=False)

    metrics = {
        "run_name": RUN_NAME,
        "best_epoch": int(best_epoch),
        "best_val_macro_ap_checkpoint": float(best_ap),
        "val_metrics_threshold_tuned": val_metrics,
        "test_metrics_threshold_tuned": test_metrics,
        "thresholds": {name: float(t) for name, t in zip(LABEL_COLS, thresholds)},
    }
    with open(run_dir / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    with open(run_dir / "config.json", "w") as f:
        json.dump(CONFIG, f, indent=2)

    print("\nFinal validation metrics:", val_metrics)
    print("Final test metrics:", test_metrics)
    print("Saved outputs to:", run_dir)
    return metrics


In [ ]:

# ============================================================
# 4. Static graph construction from training labels: included inside this notebook
# ============================================================
def topk_sparsify(A, k=5, keep_self=True):
    A = A.copy()
    L = A.shape[0]
    if keep_self:
        np.fill_diagonal(A, np.maximum(np.diag(A), 1.0))
    out = np.zeros_like(A)
    for i in range(L):
        idx = np.argsort(A[i])[-k:]
        out[i, idx] = A[i, idx]
    # make symmetric for stable GCN propagation
    out = np.maximum(out, out.T)
    if keep_self:
        np.fill_diagonal(out, 1.0)
    return out

def normalize_adj(A, eps=1e-8):
    A = A.astype(np.float32)
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg + eps))
    return (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)

def build_static_graph(Y, graph_type="pmi", top_k=5):
    Y = Y.astype(np.float32)
    N, L = Y.shape
    co = Y.T @ Y
    eps = 1e-8
    if graph_type == "cooc":
        A = co / max(N, 1)
    elif graph_type == "condprob":
        freq = Y.sum(axis=0)
        A = co / np.maximum(freq[:, None], 1.0)
        A = 0.5 * (A + A.T)
    elif graph_type == "pmi":
        p_i = Y.mean(axis=0)
        p_ij = co / max(N, 1)
        A = np.log((p_ij + eps) / (p_i[:, None] * p_i[None, :] + eps))
        A = np.maximum(A, 0.0)
    elif graph_type == "identity":
        A = np.eye(L, dtype=np.float32)
    else:
        raise ValueError(f"Unknown graph_type={graph_type}")
    A = topk_sparsify(A, k=top_k, keep_self=True)
    A = normalize_adj(A)
    return A

A_np = build_static_graph(Y_train_np, graph_type=GRAPH_TYPE, top_k=GRAPH_TOP_K)
A_static = torch.tensor(A_np, dtype=torch.float32, device=device)
print("Static adjacency:", A_static.shape, "graph_type=", GRAPH_TYPE)
print(pd.DataFrame(A_np, index=LABEL_COLS, columns=LABEL_COLS).round(3))


In [ ]:

# ============================================================
# 5. Model definition: Static GCN classifier
# ============================================================
def build_backbone(backbone_name="resnet18", pretrained=True):
    if backbone_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        m = models.resnet18(weights=weights)
        feat_dim = m.fc.in_features
        m.fc = nn.Identity()
        return m, feat_dim
    elif backbone_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        m = models.efficientnet_b0(weights=weights)
        feat_dim = m.classifier[1].in_features
        m.classifier = nn.Identity()
        return m, feat_dim
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

class GraphConvolution(nn.Module):
    def __init__(self, in_dim, out_dim, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_dim, out_dim))
        self.bias = nn.Parameter(torch.zeros(out_dim)) if bias else None
        nn.init.xavier_uniform_(self.weight)
    def forward(self, X, A):
        # X: [L, D], A: [L, L]
        out = A @ X @ self.weight
        if self.bias is not None:
            out = out + self.bias
        return out

class StaticGCNClassifier(nn.Module):
    def __init__(self, backbone_name, num_labels, A_static, label_embed_dim=256, gcn_hidden_dim=512, dropout=0.25):
        super().__init__()
        self.backbone, feat_dim = build_backbone(backbone_name, pretrained=True)
        self.A_static = A_static
        self.label_embeddings = nn.Parameter(torch.randn(num_labels, label_embed_dim) * 0.02)
        self.gcn1 = GraphConvolution(label_embed_dim, gcn_hidden_dim)
        self.gcn2 = GraphConvolution(gcn_hidden_dim, feat_dim)
        self.dropout = nn.Dropout(dropout)
        self.bias = nn.Parameter(torch.zeros(num_labels))
    def forward(self, x):
        feat = self.dropout(self.backbone(x))                    # [B, F]
        z = F.relu(self.gcn1(self.label_embeddings, self.A_static))
        z = self.dropout(z)
        label_weights = self.gcn2(z, self.A_static)              # [L, F]
        label_weights = F.normalize(label_weights, dim=-1)
        feat = F.normalize(feat, dim=-1)
        logits = feat @ label_weights.t() + self.bias
        return logits

model = StaticGCNClassifier(
    BACKBONE_NAME, NUM_LABELS, A_static,
    label_embed_dim=LABEL_EMBED_DIM,
    gcn_hidden_dim=GCN_HIDDEN_DIM,
    dropout=DROPOUT
).to(device)
print(model.__class__.__name__, "with", BACKBONE_NAME, "and", GRAPH_TYPE, "graph")
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:

# ============================================================
# Final cell: train, validate, test, and save outputs
# ============================================================
print("Run name:", RUN_NAME)
print("Output directory:", RUN_DIR)
metrics = run_training(model, RUN_DIR)
metrics
